# Enhanced VLM Candidate Classifier With Augmentation

Episode-level 6-way candidate selection using a frozen CLIP or SigLIP backbone.

This version includes light training-time image augmentation.


In [1]:
# Optional install cell if transformers is missing.
# import sys
# !{sys.executable} -m pip install transformers accelerate

In [2]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from transformers import AutoModel, AutoProcessor

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
PROJECT_ROOT = Path('/home/gyanig/catkin_ws/src/tabletop_workspace_opt')
DATA_ROOT = PROJECT_ROOT / 'data' / 'milk_candidate_cls'
SAMPLES_PATH = DATA_ROOT / 'candidate_samples.jsonl'
SPLIT_ROOT = DATA_ROOT / 'splits'
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'openai/clip-vit-base-patch32'  # Try 'google/siglip-base-patch16-224' later.
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
USE_AUGMENTATION = True
AUG_IMAGE_SIZE = 224

assert SAMPLES_PATH.exists(), SAMPLES_PATH

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

df = pd.DataFrame(load_jsonl(SAMPLES_PATH))

def base_scene_id(scene_id: str) -> str:
    for suffix in ('_top', '_side', '_lean'):
        if scene_id.endswith(suffix):
            return scene_id[: -len(suffix)]
    return scene_id

df['base_scene_id'] = df['scene_id'].map(base_scene_id)
df['text_input'] = df.apply(lambda row: f"Instruction: {row['instruction']}\nCandidate: {row['candidate_text']}", axis=1)
base_scenes = sorted(df['base_scene_id'].unique())
train_scenes = base_scenes[:-1]
val_scenes = base_scenes[-1:]
train_df = df[df['base_scene_id'].isin(train_scenes)].reset_index(drop=True)
val_df = df[df['base_scene_id'].isin(val_scenes)].reset_index(drop=True)

def build_episode_records(frame: pd.DataFrame):
    records = []
    grouped = frame.groupby('episode_id', sort=True)
    for episode_id, g in grouped:
        g = g.sort_values('candidate_id').reset_index(drop=True)
        pos = g[g['label'] == 1]
        records.append({
            'episode_id': episode_id,
            'scene_id': g.iloc[0]['scene_id'],
            'image_path': g.iloc[0]['image_path'],
            'candidate_ids': g['candidate_id'].tolist(),
            'candidate_texts': g['text_input'].tolist(),
            'correct_candidate_id': pos.iloc[0]['candidate_id'],
            'target_index': int(pos.index[0]),
        })
    return records

train_records = build_episode_records(train_df)
val_records = build_episode_records(val_df)
print('train episodes:', len(train_records), 'val episodes:', len(val_records))

train episodes: 21 val episodes: 22


In [5]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

def get_image_features(model, pixel_values):
    if hasattr(model, 'get_image_features'):
        return model.get_image_features(pixel_values=pixel_values)
    return model.vision_model(pixel_values=pixel_values).pooler_output

def get_text_features(model, input_ids, attention_mask):
    if hasattr(model, 'get_text_features'):
        return model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
    return model.text_model(input_ids=input_ids, attention_mask=attention_mask).pooler_output

def l2_normalize(x):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-6)

dummy_img = Image.open(DATA_ROOT / train_records[0]['image_path']).convert('RGB')
dummy = processor(images=dummy_img, text=train_records[0]['candidate_texts'][0], return_tensors='pt', padding=True)
with torch.no_grad():
    embed_dim = int(get_image_features(backbone, dummy['pixel_values'].to(device)).shape[-1])
embed_dim

512

In [6]:
train_image_transform = T.Compose([
    T.RandomResizedCrop(AUG_IMAGE_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
]) if USE_AUGMENTATION else None

class EpisodeDataset(Dataset):
    def __init__(self, records, image_root):
        self.records = records
        self.image_root = image_root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(self.image_root / rec['image_path']).convert('RGB')
        return {
            'image': image,
            'candidate_texts': rec['candidate_texts'],
            'target_index': rec['target_index'],
            'episode_id': rec['episode_id'],
            'candidate_ids': rec['candidate_ids'],
            'correct_candidate_id': rec['correct_candidate_id'],
        }

def collate_episode(batch, image_transform=None):
    images = [item['image'] for item in batch]
    if image_transform is not None:
        images = [image_transform(img) for img in images]
    texts = []
    for item in batch:
        texts.extend(item['candidate_texts'])
    image_inputs = processor(images=images, return_tensors='pt')
    text_inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)
    return {
        'pixel_values': image_inputs['pixel_values'],
        'input_ids': text_inputs['input_ids'],
        'attention_mask': text_inputs['attention_mask'],
        'targets': torch.tensor([item['target_index'] for item in batch], dtype=torch.long),
        'num_candidates': len(batch[0]['candidate_texts']),
    }

train_ds = EpisodeDataset(train_records, DATA_ROOT)
val_ds = EpisodeDataset(val_records, DATA_ROOT)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=lambda batch: collate_episode(batch, image_transform=train_image_transform))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=lambda batch: collate_episode(batch, image_transform=None))

In [7]:
class EpisodeRanker(nn.Module):
    def __init__(self, vlm_backbone, embed_dim):
        super().__init__()
        self.vlm_backbone = vlm_backbone
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, pixel_values, input_ids, attention_mask, num_candidates):
        batch_size = pixel_values.shape[0]
        with torch.no_grad():
            image_feat = l2_normalize(get_image_features(self.vlm_backbone, pixel_values))
            text_feat = l2_normalize(get_text_features(self.vlm_backbone, input_ids, attention_mask))
        image_feat = image_feat.unsqueeze(1).expand(batch_size, num_candidates, image_feat.shape[-1])
        text_feat = text_feat.view(batch_size, num_candidates, -1)
        fused = torch.cat([image_feat, text_feat, image_feat * text_feat], dim=-1)
        return self.scorer(fused).squeeze(-1)

model = EpisodeRanker(backbone, embed_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.scorer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [8]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    n = 0
    correct = 0
    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        logits = model(pixel_values, input_ids, attention_mask, batch['num_candidates'])
        loss = criterion(logits, targets)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        pred = logits.argmax(dim=1)
        correct += int((pred == targets).sum().item())
        n += len(targets)
        total_loss += float(loss.item()) * len(targets)
    return {'loss': total_loss / max(n, 1), 'episode_top1': correct / max(n, 1)}

history = []
best_val = -1.0
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    row = {'epoch': epoch, 'train_loss': train_metrics['loss'], 'train_episode_top1': train_metrics['episode_top1'], 'val_loss': val_metrics['loss'], 'val_episode_top1': val_metrics['episode_top1']}
    history.append(row)
    print(row)

pd.DataFrame(history)

{'epoch': 1, 'train_loss': 1.7880720354261852, 'train_episode_top1': 0.09523809523809523, 'val_loss': 1.77859674800526, 'val_episode_top1': 0.45454545454545453}
{'epoch': 2, 'train_loss': 1.7732709305627006, 'train_episode_top1': 0.3333333333333333, 'val_loss': 1.7640922286293723, 'val_episode_top1': 0.45454545454545453}
{'epoch': 3, 'train_loss': 1.7615782079242526, 'train_episode_top1': 0.19047619047619047, 'val_loss': 1.749911210753701, 'val_episode_top1': 0.45454545454545453}
{'epoch': 4, 'train_loss': 1.7443308319364275, 'train_episode_top1': 0.2857142857142857, 'val_loss': 1.734844446182251, 'val_episode_top1': 0.45454545454545453}
{'epoch': 5, 'train_loss': 1.7340698809850783, 'train_episode_top1': 0.19047619047619047, 'val_loss': 1.7148667899045078, 'val_episode_top1': 0.45454545454545453}
{'epoch': 6, 'train_loss': 1.715035075233096, 'train_episode_top1': 0.2857142857142857, 'val_loss': 1.696069067174738, 'val_episode_top1': 0.45454545454545453}
{'epoch': 7, 'train_loss': 1.68

,epoch,train_loss,train_episode_top1,val_loss,val_episode_top1
0,1,1.788072,0.095238,1.778597,0.454545
1,2,1.773271,0.333333,1.764092,0.454545
2,3,1.761578,0.190476,1.749911,0.454545
3,4,1.744331,0.285714,1.734844,0.454545
4,5,1.734070,0.190476,1.714867,0.454545
5,6,1.715035,0.285714,1.696069,0.454545
6,7,1.688499,0.380952,1.674674,0.454545
7,8,1.667805,0.380952,1.651924,0.454545
8,9,1.654280,0.285714,1.627778,0.454545
9,10,1.624687,0.476190,1.603460,0.454545
